# 09 · TabICLv2 (corre en Kaggle con GPU)

TabICLv2 (INRIA, 2026) es otro modelo fundacional tabular, como TabPFN: recibe las filas de entrenamiento como
contexto y predice en una pasada, sin entrenamiento. Tiene otra arquitectura (embeddings por columna + transformer
entre filas) y otra distribución sintética de preentrenamiento, por eso se espera que se equivoque en casos
distintos que TabPFN. Pesos abiertos: no requiere token.

Usa exactamente las mismas features y los mismos 5 folds que LightGBM/XGBoost/CatBoost (dataset privado
`ap-tp3-features`, generado desde `features.load_fold` / `features.load_full`), así que su predicción es comparable
y se puede sumar al ensamble.

In [ ]:
!pip install -q -U tabicl

In [ ]:
import glob, os, time, numpy as np, pandas as pd, torch
import warnings; warnings.filterwarnings("ignore")
from tabicl import TabICLClassifier
from importlib.metadata import version
print("tabicl", version("tabicl"), "| GPU:", torch.cuda.get_device_name(0))

DATA = os.path.dirname(glob.glob("/kaggle/input/**/fold0_tr.parquet", recursive=True)[0])
y = np.load(f"{DATA}/y.npy")
N_ESTIMATORS = 8   # valor por defecto de TabICL
SEED = 42

In [ ]:
def prepare(A, B):
    """Categoricas -> codigos enteros con el mismo diccionario en A y B."""
    A, B = A.copy(), B.copy()
    cats = [c for c in A.columns if str(A[c].dtype) == "category"]
    for c in cats:
        dic = pd.Index(pd.concat([A[c].astype(str), B[c].astype(str)]).unique())
        A[c], B[c] = dic.get_indexer(A[c].astype(str)), dic.get_indexer(B[c].astype(str))
    return A.astype("float32"), B.astype("float32"), [A.columns.get_loc(c) for c in cats]

def fit_predict(A, ya, B):
    A, B, cat_idx = prepare(A, B)
    # batch_size=1: procesa los miembros del ensamble de a uno (con 56.000 filas x 157 columnas no entran juntos en la T4)
    m = TabICLClassifier(n_estimators=N_ESTIMATORS, batch_size=1, device="cuda", random_state=SEED)   # categoricas como codigos
    m.fit(A.values, ya)
    return np.concatenate([m.predict_proba(B.values[s:s + 2000])[:, 1] for s in range(0, len(B), 2000)])

In [ ]:
out = {"oof": np.zeros(len(y), dtype="float32")}
t0 = time.time()
from sklearn.model_selection import StratifiedKFold
for f, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=0).split(y, y)):   # = features.cv_folds
    A, B = pd.read_parquet(f"{DATA}/fold{f}_tr.parquet"), pd.read_parquet(f"{DATA}/fold{f}_va.parquet")
    p = fit_predict(A, y[a], B)
    out["oof"][b] = p; out[f"valid{f}"] = p
    print(f"fold {f}: accuracy {((p > .5) == y[b]).mean():.4f}  ({(time.time() - t0) / 60:.0f} min)", flush=True)
    np.savez("tabicl.npz", **out)
print(f"Accuracy CV TabICL: {((out['oof'] > .5) == y).mean():.4f}")

In [ ]:
A, B = pd.read_parquet(f"{DATA}/full_tr.parquet"), pd.read_parquet(f"{DATA}/full_te.parquet")
out["test"] = fit_predict(A, y, B)
np.savez("tabicl.npz", **out)
print(f"test listo ({(time.time() - t0) / 60:.0f} min)")